In [ ]:
# Install latest diffusers from GitHub
!pip install -q git+https://github.com/huggingface/diffusers.git

# Install dependencies
!pip install -q -r https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/requirements_sdxl.txt
!pip install -q -U torchao peft ftfy spacy torchvision

# Download train stript
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py

In [ ]:
import os
import json
import zipfile
from google.colab import files

uploaded = files.upload()
os.makedirs("dataset", exist_ok=True)
with zipfile.ZipFile("csaba_paint_dataset.zip", "r") as zip_ref:
    zip_ref.extractall("dataset")

dataset_dir = "/content/dataset"
metadata_file = os.path.join(dataset_dir, "metadata.jsonl")

entries = []
for file in sorted(os.listdir(dataset_dir)):
    if file.endswith((".png", ".jpg", ".jpeg")):
        txt_file = os.path.splitext(file)[0] + ".txt"
        txt_path = os.path.join(dataset_dir, txt_file)

        if os.path.exists(txt_path):
            with open(txt_path, "r", encoding="utf-8") as f:
                caption = f.read().strip()
            entries.append({"file_name": file, "text": caption})

with open(metadata_file, "w", encoding="utf-8") as f:
    for entry in entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Dataset prepared! File metadata.jsonl created with {len(entries)} entries.")

In [ ]:
!accelerate launch --mixed_precision="fp16" train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --train_data_dir="/content/dataset" \
  --caption_column="text" \
  --resolution=1024 \
  --random_flip \
  --train_batch_size=1 \
  --gradient_checkpointing \
  --num_train_epochs=50 \
  --checkpointing_steps=500 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --mixed_precision="fp16" \
  --output_dir="/content/csaba_paint_lora"

In [ ]:
from google.colab import files

files.download('/content/csaba_paint_lora/pytorch_lora_weights.safetensors')

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

print("Load SDXL Base...")
pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16
)
pipe.to('cuda')

# load custom LoRA weights
pipe.load_lora_weights('/content/csaba_paint_lora/pytorch_lora_weights.safetensors')

In [ ]:
prompt = 'dragon in csaba style'

print("Generating image...")
image = pipe(
    prompt=prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    cross_attention_kwargs={"scale": 1.1},
    generator=torch.Generator("cuda").manual_seed(42)
).images[0]

image.save('dragon.png')
image